# 📚 Fine-Tuning DistilBERT for NER on MIT Restaurant Dataset

## 🛠 1. Install Required Libraries

In [1]:
!pip install -U transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.3 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=257552cdfa32e74945a179b03ec4c077a0562dc46a655061b42bccd28aca9474
  Stored in directory: /root/.cache/pip/wheels/bc/92/f0/243288f899c2eacdfa8c5f9aede4c71a9bad0ee26a01dc5ead
Successfully built seqeval
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver

In [6]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.4 MB/s eta 0:00:00


## 📥 2. Load and Preprocess the Dataset

In [2]:
import requests

def load_bio_dataset(url):
    response = requests.get(url)
    lines = response.text.strip().split('\n')

    tokens = []
    labels = []
    temp_tokens = []
    temp_labels = []

    for line in lines:
        if line == "":
            if temp_tokens:
                tokens.append(temp_tokens)
                labels.append(temp_labels)
                temp_tokens = []
                temp_labels = []
        else:
            splits = line.strip().split()
            if len(splits) == 2:
                label, token = splits
                temp_tokens.append(token)
                temp_labels.append(label)
    if temp_tokens:
        tokens.append(temp_tokens)
        labels.append(temp_labels)

    return tokens, labels

url = "https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/mit_restaurant_search_ner/train.bio"
tokens, labels = load_bio_dataset(url)

# Preview
print(tokens[0])
print(labels[0])


['2', 'start', 'restaurants', 'with', 'inside', 'dining']
['B-Rating', 'I-Rating', 'O', 'O', 'B-Amenity', 'I-Amenity']


## 📦 3. Prepare Dataset for Hugging Face Trainer

In [3]:
import pandas as pd
from datasets import Dataset, DatasetDict

data = pd.DataFrame({'tokens': tokens, 'ner_tags': labels})
hf_dataset = Dataset.from_pandas(data)

dataset = DatasetDict({
    'train': hf_dataset,
    'validation': hf_dataset,
    'test': hf_dataset
})


## 🏷️ 4. Tokenize and Align Labels

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

label_list = list(set(label for doc in labels for label in doc))
label_list.sort()
label_to_id = {label: i for i, label in enumerate(label_list)}

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    aligned_labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[label[word_idx]])
            else:
                label_ids.append(label_to_id[label[word_idx]])
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7660 [00:00<?, ? examples/s]

Map:   0%|          | 0/7660 [00:00<?, ? examples/s]

Map:   0%|          | 0/7660 [00:00<?, ? examples/s]

## 📊 5. Define Metrics

In [7]:
import numpy as np
from evaluate import load

metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


## ⚙️ 6. Set Up Training Arguments

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_model",
    eval_strategy="epoch",  # Updated parameter name
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",   # 🔥 disable reporting to wandb or any other tracking tool
)

## 🏋️ 7. Initialize Trainer and Fine-Tune

In [13]:
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer

model = AutoModelForTokenClassification.from_pretrained("distilbert-base-uncased", num_labels=len(label_list))
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-13-f51480429887>:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.390500,0.294338,0.767901,0.810129,0.788450,0.914616
2,0.289500,0.223809,0.828835,0.848490,0.838547,0.933337


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.390500,0.294338,0.767901,0.810129,0.788450,0.914616
2,0.289500,0.223809,0.828835,0.848490,0.838547,0.933337
3,0.201100,0.203721,0.843567,0.866785,0.855018,0.939613


TrainOutput(global_step=1437, training_loss=0.3932116363474953, metrics={'train_runtime': 4489.6931, 'train_samples_per_second': 5.118, 'train_steps_per_second': 0.32, 'total_flos': 117213322331568.0, 'train_loss': 0.3932116363474953, 'epoch': 3.0})

## 🎯 8. Evaluate and Save the Model

In [14]:
trainer.evaluate()

trainer.save_model("./ner_model")
tokenizer.save_pretrained("./ner_model")


('./ner_model/tokenizer_config.json',
 './ner_model/special_tokens_map.json',
 './ner_model/vocab.txt',
 './ner_model/added_tokens.json',
 './ner_model/tokenizer.json')